In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
from torchvision.datasets import ImageFolder
import os

import glob
from tqdm import tqdm
from PIL import Image


import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset,Subset
from torchvision import datasets, transforms, models
import kagglehub

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
path = path+"/dataset"
path

In [ ]:
images = glob.glob(path + "/images/*")
masks  = glob.glob(path + "/masks/*")
images.sort()
masks.sort()

In [ ]:
images[0],masks[0]

In [ ]:
df = pd.DataFrame({"images":images,"masks":masks})
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
df_train,df_test = train_test_split(df,test_size=0.2,shuffle=True,random_state=42)
csv_path = "/content"

df_train.to_csv(csv_path + "/train.csv",index=False)
df_test.to_csv(csv_path + "/test.csv",index=False)

In [ ]:
# TO DO
class CustomDataset(Dataset):
  def __init__(self,root_dir,csv_path,split="train",transform=None,target_transform=None):
    self.root_dir = root_dir ### i dont why always you guys add it even when we don't need it (i doing like laps)
    self.csv_path = csv_path
    self.split = split
    self.transform = transform
    self.target_transform = target_transform

    self.df = pd.read_csv(csv_path + f"/{split}.csv")

  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    img = self.df.iloc[index,0]
    mask = self.df.iloc[index,1]

    img = Image.open(img).convert("RGB") ## force it to be RGB
    mask = Image.open(mask).convert("L") ## Force it to be grayscale

    if self.transform:
      img = self.transform(img)
    if self.target_transform:
      mask = self.target_transform(mask)

    mask = remap_mask(mask)

    return img,mask.to(torch.long)



In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

transform_target = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224
    transforms.PILToTensor(),  # Convert to tensor
])


In [ ]:
train_dataset = CustomDataset(path,csv_path,split="train", transform=transform,target_transform = transform_target)
test_dataset = CustomDataset(path,csv_path,split="test",transform=transform,target_transform = transform_target)

In [ ]:
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

In [ ]:
img,label = next(iter(train_loader))
print(f"img shape: {img.shape}")
print(f"label shape: {label.shape}")

print(f"img unique: {img.unique().min()} --> {img.unique().max()} ")
print(f"label unique: {label.unique()}")

print(f"label dtype: {label.dtype}")

In [ ]:
import matplotlib.pyplot as plt
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img


# Display some images with their masks
for i in range(20,23):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))  # Convert (C, H, W) to (H, W, C)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,
).to(device)

In [ ]:
img,label = next(iter(train_loader))
print(f"img shape: {img.shape}")
print(f"label shape: {label.shape}")

print(f"img unique: {img.unique().min()} --> {img.unique().max()} ")
print(f"label unique: {label.unique()}")

print(f"label dtype: {label.dtype}")

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]

        outputs = model(images) # (N,8,H,W)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]

            outputs = model(images)  # Now [N, ,H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Set model to evaluation mode
model.eval()

# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx] # (C,H,W) for both img,mask

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass pred_mask (1,1,H,W)

    pred_mask = torch.argmax(pred_mask,dim=1).cpu().squeeze().numpy()  # Convert to binary mask

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original Image (Denormalized)
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Ground Truth Mask
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    # Predicted Mask
    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()


In [ ]:
# TO DO
### **🔹 Plot Training Loss Curve**
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()